In [ ]:
import os
from langchain_groq import ChatGroq
from langchain.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from IPython.display import display, HTML

# Initialize LLM model
def initialize_llm():
    llm = ChatGroq(
        temperature=0,
        groq_api_key="",
        model_name=""
    )
    return llm

# Create vector database from SQL schema files and examples
def create_vector_db():
    # Load SQL schema files and example queries
    # You should place your SQL schema files and examples in the dataset folder
    loader = DirectoryLoader("/content/dataset", glob="*.pdf", loader_cls=PyPDFLoader)
    
    # Combine all documents
    documents = loader.load()
    
    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)
    
    # Create embeddings and vector database
    embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    vector_db = Chroma.from_documents(texts, embeddings, persist_directory='./chroma_db')
    vector_db.persist()
    
    print("ChromaDB created and data saved")
    return vector_db

# Set up the question-answering chain
def setup_qa_chain(vector_db, llm):
    retriever = vector_db.as_retriever()
    
    # Create a prompt template specifically for SQL conversion
    prompt_template = """You are an expert SQL query generator. Based on the following database schema and examples, convert the natural language question into a valid SQL query.

Database Schema and Examples:
{context}

User Question: {question}

SQL Query: """
    
    PROMPT = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT}
    )
    
    return qa_chain

# Show popup for invalid or potentially harmful queries
def show_popup(message):
    display(HTML(f'<div style="background-color:#ffcccc; padding:10px; border-radius:5px;"><b>{message}</b></div>'))

# Main function
def main():
    print("Initializing Text-to-SQL Converter...")
    
    llm = initialize_llm()
    
    db_path = "./chroma_db"
    if not os.path.exists(db_path):
        # Create dataset directory if it doesn't exist
        if not os.path.exists("./dataset"):
            os.makedirs("./dataset")
            print("Created dataset directory. Please add your SQL schema files and examples there.")
            print("Then run this script again.")
            return
        
        vector_db = create_vector_db()
    else:
        embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        vector_db = Chroma(persist_directory=db_path, embedding_function=embeddings)
    
    qa_chain = setup_qa_chain(vector_db, llm)
    
    # Keywords that might indicate potentially harmful or invalid SQL queries
    keywords = ["drop", "delete", "truncate", "alter", "update without where", "insert into admin"]
    
    print("\nText-to-SQL Converter is ready!")
    print("Type 'exit' to quit")
    
    while True:
        query = input("\nEnter your question: ")
        
        if query.lower() in ["exit", "quit", "bye"]:
            print("Exiting Text-to-SQL Converter. Goodbye!")
            break
        
        # Check for potentially harmful keywords
        if any(keyword in query.lower() for keyword in keywords):
            show_popup("Warning: Your query contains potentially harmful SQL operations.")
        
        # Generate SQL query
        sql_query = qa_chain.run(query)
        
        print(f"\nGenerated SQL Query:\n{sql_query}")
        
        # Option to explain the query
        explain = input("\nWould you like an explanation of this query? (y/n): ")
        if explain.lower() == 'y':
            explanation_prompt = f"Explain this SQL query in simple terms: {sql_query}"
            explanation = llm.invoke(explanation_prompt)
            print(f"\nExplanation:\n{explanation.content}")

if __name__ == "__main__":
    main()